# Python OOP Patterns

**Goal:** Classes, `self`, inheritance, ABCs, and dunder methods — how Python OOP differs from Go structs and TS classes.

**Prereq:** Complete [01_data_structures.ipynb](./01_data_structures.ipynb) first.

## Classes and `self`

**Go/TS comparison:** Go uses structs + methods with receivers (`func (d Dog) Speak()`). TS uses `class` with implicit `this`. Python uses `class` with **explicit `self`** — you must pass it as the first param to every method. This is Python's biggest OOP surprise.

In [ ]:
class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed
    
    def speak(self):
        return f"{self.name} says woof!"

my_dog = Dog("Rex", "German Shepherd")
print(my_dog.speak())
print(f"Name: {my_dog.name}, Breed: {my_dog.breed}")

In [ ]:
# Experiment: self is just the instance passed explicitly
# These two calls are identical:
print(my_dog.speak())        # Python passes self automatically
print(Dog.speak(my_dog))     # You can pass it manually — same result

### ✍️ In Your Own Words

Why does Python require explicit `self` while Go uses receivers and TS uses implicit `this`? Write your answer here.

## Dunder Methods (Magic Methods)

**Go/TS comparison:** Like implementing `String()` or the `Stringer` interface in Go, or `toString()` in TS. Python uses double-underscore methods (`__str__`, `__repr__`, `__len__`, `__eq__`) to hook into built-in behaviors like `print()`, `len()`, and `==`.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __str__(self):
        """Used by print() and str() — human-friendly"""
        return f"({self.x}, {self.y})"
    
    def __repr__(self):
        """Used by the REPL and debugging — unambiguous"""
        return f"Point({self.x}, {self.y})"
    
    def __eq__(self, other):
        """Used by == operator"""
        return self.x == other.x and self.y == other.y

p1 = Point(3, 4)
p2 = Point(3, 4)
p3 = Point(1, 2)

print(f"str:  {p1}")          # (3, 4)
print(f"repr: {repr(p1)}")    # Point(3, 4)
print(f"p1 == p2: {p1 == p2}")  # True
print(f"p1 == p3: {p1 == p3}")  # False

In [ ]:
class Deck:
    def __init__(self):
        suits = ['♠', '♥', '♦', '♣']
        ranks = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']
        self.cards = [f"{r}{s}" for s in suits for r in ranks]
    
    def __len__(self):
        """Makes len(deck) work"""
        return len(self.cards)
    
    def __getitem__(self, index):
        """Makes deck[0], deck[-1], and iteration work"""
        return self.cards[index]

deck = Deck()
print(f"Cards in deck: {len(deck)}")
print(f"First card: {deck[0]}")
print(f"Last card: {deck[-1]}")
print(f"First 5: {deck[:5]}")

In [ ]:
# Experiment: __add__ for custom + operator
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

p1 = Point(1, 2)
p2 = Point(3, 4)
p3 = p1 + p2  # Calls p1.__add__(p2)
print(f"{p1} + {p2} = {p3}")  # Point(1, 2) + Point(3, 4) = Point(4, 6)

### ✍️ In Your Own Words

What's the difference between `__str__` and `__repr__`? When does Python use each? Write your answer here.

## Inheritance

**Go/TS comparison:** Go has **no inheritance** (composition via embedding only). TS has `extends`. Python has full single and multiple inheritance with `super()` for calling parent methods.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    
    def speak(self):
        raise NotImplementedError("Subclasses must implement speak()")

class Cat(Animal):
    def speak(self):
        return f"{self.name} says meow!"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)  # Call parent's __init__
        self.breed = breed
    
    def speak(self):
        return f"{self.name} the {self.breed} says woof!"

cat = Cat("Whiskers")
dog = Dog("Rex", "Lab")
print(cat.speak())
print(dog.speak())

In [ ]:
# Experiment: isinstance() and issubclass()
print(f"dog is Animal? {isinstance(dog, Animal)}")      # True
print(f"dog is Dog?    {isinstance(dog, Dog)}")          # True
print(f"dog is Cat?    {isinstance(dog, Cat)}")          # False
print(f"Dog subclass of Animal? {issubclass(Dog, Animal)}")  # True

### ✍️ In Your Own Words

When would you use inheritance vs composition in Python? How does this compare to Go's approach? Write your answer here.

## Abstract Base Classes (ABCs)

**Go/TS comparison:** Like Go interfaces but enforced differently. Go interfaces are satisfied **implicitly** (duck typing at compile time). Python ABCs must be **explicitly inherited** and use `@abstractmethod`. If you forget to implement a method, you get an error when you try to create an instance.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass
    
    @abstractmethod
    def perimeter(self) -> float:
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        return math.pi * self.radius ** 2
    
    def perimeter(self):
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)

c = Circle(5)
r = Rectangle(3, 4)
print(f"Circle:    area={c.area():.2f}, perimeter={c.perimeter():.2f}")
print(f"Rectangle: area={r.area():.2f}, perimeter={r.perimeter():.2f}")

In [ ]:
# Experiment: try to instantiate the ABC directly
try:
    s = Shape()
except TypeError as e:
    print(f"Error: {e}")
# Can't instantiate abstract class Shape with abstract methods area, perimeter

### ✍️ In Your Own Words

How do Python ABCs compare to Go interfaces? Which approach do you prefer and why? Write your answer here.

## Properties and Access Control

**Go/TS comparison:** Go has exported/unexported (capital letter). TS has `private`/`public`/`protected`. Python has **no real private** — `_name` is a convention ("please don't touch"), `__name` triggers name mangling but isn't truly private. The culture is "we're all consenting adults."

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance  # Convention: "private" (but not enforced)
    
    @property
    def balance(self):
        """Getter — called when you access .balance"""
        return self._balance
    
    @balance.setter
    def balance(self, amount):
        """Setter — called when you assign to .balance"""
        if amount < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = amount
    
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        return self._balance

acct = BankAccount("Kyle", 100)
print(f"Balance: {acct.balance}")  # Uses @property getter
acct.deposit(50)
print(f"After deposit: {acct.balance}")

try:
    acct.balance = -100  # Uses @balance.setter — raises error
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# Experiment: _balance is not actually private — you CAN access it
print(f"Direct access: {acct._balance}")  # Works fine!
acct._balance = -9999  # No error — Python trusts you
print(f"After bypass: {acct._balance}")   # -9999
# This is why Python says "we're all consenting adults"

### ✍️ In Your Own Words

How does Python's approach to privacy differ from Go and TS? What are the trade-offs? Write your answer here.

## Recap

- ✅ Explicit `self` — first param in every method
- ✅ `__init__` is the constructor, `__str__`/`__repr__` for string output
- ✅ Dunder methods hook into built-ins (`len()`, `[]`, `+`, `==`)
- ✅ `super().__init__()` for parent initialization
- ✅ ABCs enforce interfaces — explicit inheritance + `@abstractmethod`
- ✅ `@property` for getters/setters
- ✅ `_name` convention, not enforcement — "we're all consenting adults"

**Next:** [03_async_basics.ipynb](./03_async_basics.ipynb)